# 02 — Data Cleaning

## Olist E-Commerce Business Analytics

### Objective

This notebook creates clean and analysis-ready versions of the nine Olist datasets based on the issues documented during the data-quality assessment.

The cleaning process covers:

- Standardising column names and text values
- Converting timestamp columns to datetime
- Handling missing values transparently
- Removing exact geolocation duplicates
- Adding missing category translations
- Flagging inconsistent timestamps
- Correcting invalid numerical values
- Preserving relevant incomplete transactions
- Validating every transformation

The original files in `data/raw/` will never be modified. All transformations will be applied to DataFrame copies and exported to `data/processed/`.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

RAW_DATA_DIR = Path("../data/raw").resolve()
PROCESSED_DATA_DIR = Path("../data/processed").resolve()

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Raw data directory:")
print(RAW_DATA_DIR)

print("\nProcessed data directory:")
print(PROCESSED_DATA_DIR)

Raw data directory:
/Users/saadmaher/Desktop/Data Science/Project portfolio/olist-ecommerce-business-analytics/data/raw

Processed data directory:
/Users/saadmaher/Desktop/Data Science/Project portfolio/olist-ecommerce-business-analytics/data/processed


## 1. Load the Raw Data

Fresh copies of the original CSV files are loaded to ensure that the cleaning workflow is reproducible and does not depend on variables created in the previous notebook.

In [2]:
dataset_files = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

raw_datasets = {
    dataset_name: pd.read_csv(RAW_DATA_DIR / filename)
    for dataset_name, filename in dataset_files.items()
}

cleaned_datasets = {
    dataset_name: dataframe.copy()
    for dataset_name, dataframe in raw_datasets.items()
}

print(f"Raw datasets loaded: {len(raw_datasets)}")
print(f"Working copies created: {len(cleaned_datasets)}")

Raw datasets loaded: 9
Working copies created: 9


***Observation:*** All nine raw datasets were loaded successfully, and independent working copies were created for cleaning.

***Data-preservation decision:*** All transformations will be applied only to `cleaned_datasets`. The DataFrames stored in `raw_datasets` will remain unchanged for validation and reconciliation.

## 2. Cleaning Log

Every cleaning transformation is documented with the affected dataset, action, reason, and number of affected records. This creates an auditable record of how the processed data differs from the raw data.

In [3]:
cleaning_log = []

def log_cleaning_action(
    dataset,
    action,
    reason,
    affected_records
):
    cleaning_log.append({
        "dataset": dataset,
        "action": action,
        "reason": reason,
        "affected_records": int(affected_records)
    })

print("Cleaning log initialised.")

Cleaning log initialised.


***Methodology:*** Cleaning actions will be logged immediately after they are performed. No row or value will be changed silently.

## 3. Standardise Column Names

Column names should be clear, correctly spelled, and consistent across the project. The original Olist products table contains two misspelled length-related column names.

In [4]:
product_column_corrections = {
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length"
}

columns_before = cleaned_datasets["products"].columns.tolist()

cleaned_datasets["products"].rename(
    columns=product_column_corrections,
    inplace=True
)

columns_after = cleaned_datasets["products"].columns.tolist()

log_cleaning_action(
    dataset="products",
    action="Corrected two misspelled column names",
    reason="Improve naming accuracy and consistency",
    affected_records=2
)

print("Columns before:")
print(columns_before)

print("\nColumns after:")
print(columns_after)

Columns before:
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

Columns after:
['product_id', 'product_category_name', 'product_name_length', 'product_description_length', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']


***Observation:*** The two misspelled product columns were renamed from `product_name_lenght` and `product_description_lenght` to `product_name_length` and `product_description_length`.

***Validation:*** Only column labels were changed; no product values or records were modified.

## 4. Convert Timestamp Columns

Timestamp columns were imported as text. Converting them to Pandas datetime values enables chronological validation, delivery-duration calculations, monthly analysis, cohorts, and retention analysis.

In [5]:
datetime_columns = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "order_items": [
        "shipping_limit_date"
    ],
    "reviews": [
        "review_creation_date",
        "review_answer_timestamp"
    ]
}

datetime_conversion_records = []

for dataset_name, columns in datetime_columns.items():
    dataframe = cleaned_datasets[dataset_name]

    for column in columns:
        non_null_before = dataframe[column].notna().sum()

        dataframe[column] = pd.to_datetime(
            dataframe[column],
            errors="coerce"
        )

        non_null_after = dataframe[column].notna().sum()
        newly_missing = non_null_before - non_null_after

        datetime_conversion_records.append({
            "dataset": dataset_name,
            "column": column,
            "data_type_after": str(dataframe[column].dtype),
            "non_null_values": non_null_after,
            "newly_missing_values": newly_missing
        })

    log_cleaning_action(
        dataset=dataset_name,
        action=f"Converted {len(columns)} timestamp column(s) to datetime",
        reason="Enable reliable time-based analysis",
        affected_records=len(dataframe)
    )

datetime_conversion_summary = pd.DataFrame(
    datetime_conversion_records
)

datetime_conversion_summary

,dataset,column,data_type_after,non_null_values,newly_missing_values
0,orders,order_purchase_timestamp,datetime64[ns],99441,0
1,orders,order_approved_at,datetime64[ns],99281,0
2,orders,order_delivered_carrier_date,datetime64[ns],97658,0
3,orders,order_delivered_customer_date,datetime64[ns],96476,0
4,orders,order_estimated_delivery_date,datetime64[ns],99441,0
5,order_items,shipping_limit_date,datetime64[ns],112650,0
6,reviews,review_creation_date,datetime64[ns],99224,0
7,reviews,review_answer_timestamp,datetime64[ns],99224,0


***Observation:*** All eight timestamp columns were successfully converted to `datetime64[ns]`. No additional missing values were created during conversion, confirming that every original non-null timestamp could be parsed correctly.

***Cleaning decision:*** Existing missing timestamps were preserved because they are primarily associated with incomplete order lifecycle stages. They will be handled according to the requirements of each analysis rather than globally imputed.

***Validation:*** The number of non-null values after conversion matches the original non-null counts recorded during the data-quality assessment.

## 5. Handle Missing Review Text

Review titles and messages are optional fields. Missing text will not be imputed with artificial content or used as a reason to remove valid review scores. Instead, indicator columns will distinguish reviews with and without written feedback.

In [6]:
reviews_clean = cleaned_datasets["reviews"]

reviews_clean["has_comment_title"] = (
    reviews_clean["review_comment_title"].notna()
)

reviews_clean["has_comment_message"] = (
    reviews_clean["review_comment_message"].notna()
)

reviews_clean["has_written_feedback"] = (
    reviews_clean["has_comment_title"]
    | reviews_clean["has_comment_message"]
)

review_text_summary = pd.DataFrame({
    "metric": [
        "Reviews with a title",
        "Reviews with a message",
        "Reviews with any written feedback",
        "Reviews with rating only"
    ],
    "review_count": [
        reviews_clean["has_comment_title"].sum(),
        reviews_clean["has_comment_message"].sum(),
        reviews_clean["has_written_feedback"].sum(),
        (~reviews_clean["has_written_feedback"]).sum()
    ]
})

review_text_summary["review_rate"] = (
    review_text_summary["review_count"]
    / len(reviews_clean)
)

log_cleaning_action(
    dataset="reviews",
    action="Created review-text availability indicators",
    reason="Preserve valid ratings while distinguishing written feedback",
    affected_records=len(reviews_clean)
)

review_text_summary.style.format({
    "review_count": "{:,.0f}",
    "review_rate": "{:.2%}"
})

,metric,review_count,review_rate
0,Reviews with a title,"11,568",11.66%
1,Reviews with a message,"40,977",41.30%
2,Reviews with any written feedback,"42,706",43.04%
3,Reviews with rating only,"56,518",56.96%


***Observation:*** Only 43.04% of reviews contain some form of written feedback, while 56.96% contain a rating only. Review messages are more common than review titles, appearing in 41.30% and 11.66% of reviews respectively.

***Cleaning decision:*** Missing review text was retained as null because its absence is expected rather than erroneous. Boolean indicators were added so written-feedback analysis can be performed without altering or removing valid numerical ratings.

***Validation:*** All 99,224 review records and their original review scores were preserved.

## 6. Clean Product Categories

Missing product categories cannot be inferred reliably from the available attributes. They will therefore be assigned an explicit `unknown` category. Two documented English translations will also be added for categories omitted from the original translation table.

In [7]:
products_clean = cleaned_datasets["products"]
translations_clean = cleaned_datasets["category_translation"]

missing_categories_before = (
    products_clean["product_category_name"].isna().sum()
)

products_clean["product_category_name"] = (
    products_clean["product_category_name"]
    .fillna("unknown")
)

additional_translations = pd.DataFrame({
    "product_category_name": [
        "pc_gamer",
        "portateis_cozinha_e_preparadores_de_alimentos",
        "unknown"
    ],
    "product_category_name_english": [
        "pc_gaming",
        "portable_kitchen_and_food_preparation_appliances",
        "unknown"
    ]
})

translations_clean = pd.concat(
    [
        translations_clean,
        additional_translations[
            ~additional_translations["product_category_name"].isin(
                translations_clean["product_category_name"]
            )
        ]
    ],
    ignore_index=True
)

cleaned_datasets["category_translation"] = translations_clean

remaining_untranslated_categories = (
    set(products_clean["product_category_name"])
    - set(translations_clean["product_category_name"])
)

log_cleaning_action(
    dataset="products",
    action="Replaced missing product categories with 'unknown'",
    reason="Preserve products without inventing unsupported categories",
    affected_records=missing_categories_before
)

log_cleaning_action(
    dataset="category_translation",
    action="Added two missing translations and an unknown-category label",
    reason="Provide complete English category coverage",
    affected_records=3
)

print("Missing categories replaced:", missing_categories_before)
print("Translation rows after cleaning:", len(translations_clean))
print(
    "Remaining untranslated categories:",
    len(remaining_untranslated_categories)
)

Missing categories replaced: 610
Translation rows after cleaning: 74
Remaining untranslated categories: 0


***Observation:*** A total of 610 products had no category, and two existing Portuguese categories were absent from the original translation table.

***Cleaning decision:*** Missing product categories were labelled `unknown` rather than inferred. Documented English labels were added for the two untranslated categories, and an `unknown` translation was included to ensure complete translation coverage.

***Validation:*** All product category values now have a corresponding entry in the category translation table.

### 6.1 Product Attribute Cleaning

Product characteristics are required for selected category and logistics analyses. Unsupported imputation could distort results, so missing descriptive and physical attributes will be preserved. Non-positive weights will be treated as invalid and converted to null values.

In [8]:
product_attribute_columns = [
    "product_name_length",
    "product_description_length",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

invalid_weight_mask = (
    products_clean["product_weight_g"].notna()
    & (products_clean["product_weight_g"] <= 0)
)

invalid_weights_count = invalid_weight_mask.sum()

products_clean.loc[
    invalid_weight_mask,
    "product_weight_g"
] = np.nan

products_clean["has_complete_product_attributes"] = (
    products_clean[product_attribute_columns]
    .notna()
    .all(axis=1)
)

product_attribute_summary = pd.DataFrame({
    "column": product_attribute_columns,
    "missing_after_cleaning": [
        products_clean[column].isna().sum()
        for column in product_attribute_columns
    ]
})

log_cleaning_action(
    dataset="products",
    action="Converted non-positive product weights to missing values",
    reason="Prevent invalid weights from distorting logistics analysis",
    affected_records=invalid_weights_count
)

log_cleaning_action(
    dataset="products",
    action="Created product-attribute completeness indicator",
    reason="Allow analyses to filter products requiring complete attributes",
    affected_records=len(products_clean)
)

print("Invalid weights converted to missing:", invalid_weights_count)

product_attribute_summary

Invalid weights converted to missing: 4


,column,missing_after_cleaning
0,product_name_length,610
1,product_description_length,610
2,product_photos_qty,610
3,product_weight_g,6
4,product_length_cm,2
5,product_height_cm,2
6,product_width_cm,2


***Observation:*** Four invalid non-positive product weights were converted to null, increasing missing product weights from two to six. The remaining missing attributes were preserved because no reliable values can be inferred from the available data.

***Cleaning decision:*** Product records will remain available for revenue and category analysis. Analyses requiring physical attributes will use the `has_complete_product_attributes` indicator to select appropriate records.

***Validation:*** All 32,951 product records were retained, and no valid product measurements were modified.

## 7. Clean the Geolocation Dataset

The geolocation table contains exact duplicate rows. Removing these duplicates reduces memory usage and prevents identical coordinate records from being counted repeatedly. Legitimate multiple coordinates associated with the same ZIP-code prefix will be preserved.

In [9]:
geolocation_clean = cleaned_datasets["geolocation"]

rows_before = len(geolocation_clean)
duplicate_rows = geolocation_clean.duplicated().sum()

geolocation_clean = (
    geolocation_clean
    .drop_duplicates()
    .reset_index(drop=True)
)

rows_after = len(geolocation_clean)

cleaned_datasets["geolocation"] = geolocation_clean

remaining_exact_duplicates = (
    geolocation_clean.duplicated().sum()
)

log_cleaning_action(
    dataset="geolocation",
    action="Removed exact duplicate rows",
    reason="Reduce redundancy and prevent repeated coordinate records",
    affected_records=duplicate_rows
)

print("Rows before cleaning:", f"{rows_before:,}")
print("Exact duplicates removed:", f"{duplicate_rows:,}")
print("Rows after cleaning:", f"{rows_after:,}")
print(
    "Remaining exact duplicates:",
    f"{remaining_exact_duplicates:,}"
)

Rows before cleaning: 1,000,163
Exact duplicates removed: 261,831
Rows after cleaning: 738,332
Remaining exact duplicates: 0


***Observation:*** The geolocation dataset contained 261,831 exact duplicate rows, representing 26.18% of the original records.

***Cleaning decision:*** Exact duplicates were removed from the working copy, reducing the dataset to 738,332 rows. Non-identical records sharing the same ZIP-code prefix were preserved because they may represent legitimate geographic variation.

***Validation:*** No exact duplicate rows remain in the cleaned geolocation table, and the original raw dataset was not modified.

## 8. Investigate Payment Anomalies

The data-quality assessment identified a small number of zero payment values, zero installments, and undefined payment methods. These records must be examined before deciding whether they represent errors, valid marketplace behaviour, or incomplete transactions.

In [10]:
payments_clean = cleaned_datasets["payments"]

payment_anomaly_mask = (
    (payments_clean["payment_value"] == 0)
    | (payments_clean["payment_installments"] == 0)
    | (payments_clean["payment_type"] == "not_defined")
)

payment_anomalies = (
    payments_clean.loc[
        payment_anomaly_mask,
        [
            "order_id",
            "payment_sequential",
            "payment_type",
            "payment_installments",
            "payment_value"
        ]
    ]
    .sort_values(
        [
            "payment_type",
            "payment_value",
            "payment_installments"
        ]
    )
    .reset_index(drop=True)
)

print("Payment anomaly rows:", len(payment_anomalies))

payment_anomalies

Payment anomaly rows: 11


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
1,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94
2,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.00
3,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.00
4,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.00
5,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.00
6,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.00
7,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.00
8,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.00
9,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.00


***Observation:*** Eleven payment records contain at least one anomaly. Two positive credit-card payments report zero installments, three zero-value payments have an undefined payment method, and six zero-value records use vouchers.

***Cleaning decision:*** Zero-value payment records will be retained because they may be valid components of multi-payment orders. Undefined payment types will be standardised to `unknown`, while zero installments will be converted to null because zero is not a valid installment count. Indicator columns will preserve the original anomaly information.

In [11]:
payments_clean["is_zero_payment"] = (
    payments_clean["payment_value"] == 0
)

payments_clean["has_invalid_installments"] = (
    payments_clean["payment_installments"] == 0
)

payments_clean["has_undefined_payment_type"] = (
    payments_clean["payment_type"] == "not_defined"
)

payments_clean["has_payment_anomaly"] = (
    payments_clean["is_zero_payment"]
    | payments_clean["has_invalid_installments"]
    | payments_clean["has_undefined_payment_type"]
)

invalid_installment_count = (
    payments_clean["has_invalid_installments"].sum()
)

undefined_payment_count = (
    payments_clean["has_undefined_payment_type"].sum()
)

payments_clean.loc[
    payments_clean["has_invalid_installments"],
    "payment_installments"
] = np.nan

payments_clean["payment_installments"] = (
    payments_clean["payment_installments"].astype("Int64")
)

payments_clean["payment_type"] = (
    payments_clean["payment_type"]
    .replace({"not_defined": "unknown"})
)

log_cleaning_action(
    dataset="payments",
    action="Converted zero installment counts to missing values",
    reason="Zero is not a valid number of payment installments",
    affected_records=invalid_installment_count
)

log_cleaning_action(
    dataset="payments",
    action="Standardised undefined payment types to 'unknown'",
    reason="Use a clear and consistent missing-category label",
    affected_records=undefined_payment_count
)

log_cleaning_action(
    dataset="payments",
    action="Created payment anomaly indicators",
    reason="Preserve unusual payment information for investigation",
    affected_records=payments_clean["has_payment_anomaly"].sum()
)

print(
    "Remaining zero installments:",
    (payments_clean["payment_installments"] == 0).sum()
)

print(
    "Remaining not_defined payment types:",
    (payments_clean["payment_type"] == "not_defined").sum()
)

print(
    "Flagged payment anomalies:",
    payments_clean["has_payment_anomaly"].sum()
)

Remaining zero installments: 0
Remaining not_defined payment types: 0
Flagged payment anomalies: 11


***Observation:*** Eleven payment records were flagged with at least one anomaly. After cleaning, no zero installment counts or `not_defined` payment types remain.

***Cleaning decision:*** Zero installment values were converted to null, and undefined payment types were standardised to `unknown`. Zero-value payment records were retained and flagged because they may represent valid components of multi-payment transactions.

***Validation:*** All 103,886 payment records were preserved, and the 11 anomalous records remain identifiable through the payment-quality indicator columns.

## 9. Handle Timestamp Inconsistencies

A small number of orders contain timestamps that do not follow the expected purchase-to-delivery sequence. The source values will be preserved, but quality indicators will identify records that should not be used in delivery-duration calculations.

In [12]:
orders_clean = cleaned_datasets["orders"]

orders_clean["carrier_before_purchase"] = (
    orders_clean["order_delivered_carrier_date"]
    < orders_clean["order_purchase_timestamp"]
)

orders_clean["delivery_before_purchase"] = (
    orders_clean["order_delivered_customer_date"]
    < orders_clean["order_purchase_timestamp"]
)

orders_clean["delivery_before_carrier"] = (
    orders_clean["order_delivered_customer_date"]
    < orders_clean["order_delivered_carrier_date"]
)

orders_clean["has_invalid_timestamp_sequence"] = (
    orders_clean["carrier_before_purchase"]
    | orders_clean["delivery_before_purchase"]
    | orders_clean["delivery_before_carrier"]
)

orders_clean["has_complete_delivery_timestamps"] = (
    orders_clean[
        [
            "order_purchase_timestamp",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date"
        ]
    ]
    .notna()
    .all(axis=1)
)

orders_clean["is_valid_for_delivery_analysis"] = (
    (orders_clean["order_status"] == "delivered")
    & orders_clean["has_complete_delivery_timestamps"]
    & ~orders_clean["has_invalid_timestamp_sequence"]
)

timestamp_flag_summary = pd.DataFrame({
    "quality_flag": [
        "Carrier before purchase",
        "Delivery before purchase",
        "Delivery before carrier",
        "Any invalid timestamp sequence",
        "Valid for delivery analysis"
    ],
    "order_count": [
        orders_clean["carrier_before_purchase"].sum(),
        orders_clean["delivery_before_purchase"].sum(),
        orders_clean["delivery_before_carrier"].sum(),
        orders_clean["has_invalid_timestamp_sequence"].sum(),
        orders_clean["is_valid_for_delivery_analysis"].sum()
    ]
})

log_cleaning_action(
    dataset="orders",
    action="Created timestamp-quality indicators",
    reason="Exclude invalid sequences without deleting complete orders",
    affected_records=orders_clean[
        "has_invalid_timestamp_sequence"
    ].sum()
)

timestamp_flag_summary.style.format({
    "order_count": "{:,.0f}"
})

,quality_flag,order_count
0,Carrier before purchase,166
1,Delivery before purchase,0
2,Delivery before carrier,23
3,Any invalid timestamp sequence,189
4,Valid for delivery analysis,"96,281"


***Observation:*** A total of 189 orders contain invalid timestamp sequences: 166 were recorded as handed to the carrier before purchase, and 23 were recorded as delivered before carrier handoff. No orders were delivered before purchase.

***Cleaning decision:*** The original timestamps were preserved and quality flags were added instead of attempting unsupported corrections. A total of 96,281 delivered orders have complete and chronologically valid timestamps and are eligible for delivery-performance analysis.

***Validation:*** No order records or timestamp values were deleted or overwritten. The `is_valid_for_delivery_analysis` indicator provides a reproducible analytical filter.

In [14]:
timestamp_flag_summary.style.format({
    "order_count": "{:,.0f}"
})

,quality_flag,order_count
0,Carrier before purchase,166
1,Delivery before purchase,0
2,Delivery before carrier,23
3,Any invalid timestamp sequence,189
4,Valid for delivery analysis,"96,281"


## 10. Standardise Categorical Text

Categorical values should use consistent spacing and letter case. Cities and category labels will use lowercase text, state codes will use uppercase text, and status and payment labels will use lowercase text.

In [15]:
text_standardisation_rules = {
    "customers": {
        "customer_city": "lower",
        "customer_state": "upper"
    },
    "sellers": {
        "seller_city": "lower",
        "seller_state": "upper"
    },
    "geolocation": {
        "geolocation_city": "lower",
        "geolocation_state": "upper"
    },
    "orders": {
        "order_status": "lower"
    },
    "payments": {
        "payment_type": "lower"
    },
    "products": {
        "product_category_name": "lower"
    },
    "category_translation": {
        "product_category_name": "lower",
        "product_category_name_english": "lower"
    }
}

text_cleaning_records = []

for dataset_name, column_rules in text_standardisation_rules.items():
    dataframe = cleaned_datasets[dataset_name]
    dataset_changes = 0

    for column, case_rule in column_rules.items():
        original_values = dataframe[column].astype("string")

        cleaned_values = original_values.str.strip()

        if case_rule == "lower":
            cleaned_values = cleaned_values.str.lower()
        elif case_rule == "upper":
            cleaned_values = cleaned_values.str.upper()

        changed_values = (
            original_values.ne(cleaned_values)
            .fillna(False)
            .sum()
        )

        dataframe[column] = cleaned_values
        dataset_changes += changed_values

        text_cleaning_records.append({
            "dataset": dataset_name,
            "column": column,
            "case_rule": case_rule,
            "changed_values": changed_values
        })

    log_cleaning_action(
        dataset=dataset_name,
        action="Standardised categorical text formatting",
        reason="Ensure consistent case and remove surrounding whitespace",
        affected_records=dataset_changes
    )

text_cleaning_summary = pd.DataFrame(text_cleaning_records)

text_cleaning_summary

,dataset,column,case_rule,changed_values
0,customers,customer_city,lower,0
1,customers,customer_state,upper,0
2,sellers,seller_city,lower,0
3,sellers,seller_state,upper,0
4,geolocation,geolocation_city,lower,1
5,geolocation,geolocation_state,upper,0
6,orders,order_status,lower,0
7,payments,payment_type,lower,0
8,products,product_category_name,lower,0
9,category_translation,product_category_name,lower,0


***Observation:*** Almost all categorical fields already followed the intended formatting conventions. Only one geolocation city value changed after trimming whitespace and enforcing lowercase text.

***Cleaning decision:*** Consistent formatting rules were applied across customer, seller, geolocation, order, payment, and product-category fields even where no changes were required.

***Validation:*** State codes remain uppercase, while cities, order statuses, payment types, and category labels remain lowercase. Only one source value was modified.

## 11. Standardise ZIP-Code Prefixes

Brazilian ZIP-code prefixes identify geographic areas and should not be treated as numerical measurements. Converting them to five-character text values preserves leading zeros and creates consistent join keys.

In [16]:
zip_code_columns = {
    "customers": "customer_zip_code_prefix",
    "sellers": "seller_zip_code_prefix",
    "geolocation": "geolocation_zip_code_prefix"
}

zip_cleaning_records = []

for dataset_name, column in zip_code_columns.items():
    dataframe = cleaned_datasets[dataset_name]

    original_values = dataframe[column].astype("string")

    cleaned_values = (
        original_values
        .str.replace(r"\.0$", "", regex=True)
        .str.zfill(5)
    )

    changed_values = (
        original_values.ne(cleaned_values)
        .fillna(False)
        .sum()
    )

    dataframe[column] = cleaned_values

    zip_cleaning_records.append({
        "dataset": dataset_name,
        "column": column,
        "changed_values": changed_values,
        "minimum_length": dataframe[column].str.len().min(),
        "maximum_length": dataframe[column].str.len().max()
    })

    log_cleaning_action(
        dataset=dataset_name,
        action="Converted ZIP-code prefixes to five-character text",
        reason="Preserve leading zeros and prevent numerical treatment",
        affected_records=changed_values
    )

zip_cleaning_summary = pd.DataFrame(zip_cleaning_records)

zip_cleaning_summary

,dataset,column,changed_values,minimum_length,maximum_length
0,customers,customer_zip_code_prefix,23995,5,5
1,sellers,seller_zip_code_prefix,1027,5,5
2,geolocation,geolocation_zip_code_prefix,158832,5,5


***Observation:*** A substantial number of ZIP-code prefixes required leading-zero restoration after conversion from numeric values. All customer, seller, and geolocation prefixes now contain exactly five characters.

***Cleaning decision:*** ZIP-code prefixes were stored as text because they represent geographic identifiers rather than numerical quantities.

***Validation:*** The minimum and maximum ZIP-code lengths are both five across all three datasets, confirming consistent formatting for future geographic joins.

## 12. Handle Incomplete Order Relationships

A small number of orders do not have corresponding item, payment, or review records. These orders will be retained, while Boolean indicators will identify which analyses they can support.

In [19]:
orders_clean["has_order_items"] = (
    orders_clean["order_id"].isin(
        cleaned_datasets["order_items"]["order_id"]
    )
)

orders_clean["has_payment"] = (
    orders_clean["order_id"].isin(
        cleaned_datasets["payments"]["order_id"]
    )
)

orders_clean["has_review"] = (
    orders_clean["order_id"].isin(
        cleaned_datasets["reviews"]["order_id"]
    )
)

relationship_flag_summary = pd.DataFrame({
    "relationship": [
        "Order items",
        "Payment",
        "Review"
    ],
    "orders_with_record": [
        orders_clean["has_order_items"].sum(),
        orders_clean["has_payment"].sum(),
        orders_clean["has_review"].sum()
    ],
    "orders_without_record": [
        (~orders_clean["has_order_items"]).sum(),
        (~orders_clean["has_payment"]).sum(),
        (~orders_clean["has_review"]).sum()
    ]
})

relationship_flag_summary["coverage_rate"] = (
    relationship_flag_summary["orders_with_record"]
    / len(orders_clean)
)

log_cleaning_action(
    dataset="orders",
    action="Created order relationship-coverage indicators",
    reason="Preserve incomplete orders while supporting safe analytical filters",
    affected_records=len(orders_clean)
)

relationship_flag_summary.style.format({
    "orders_with_record": "{:,.0f}",
    "orders_without_record": "{:,.0f}",
    "coverage_rate": "{:.2%}"
})

,relationship,orders_with_record,orders_without_record,coverage_rate
0,Order items,"98,666",775,99.22%
1,Payment,"99,440",1,100.00%
2,Review,"98,673",768,99.23%


***Observation:*** Relationship coverage is very high: 99.22% of orders have items, all but one order have payment records, and 99.23% have reviews. Missing reviews are expected because customer feedback is optional.

***Cleaning decision:*** All orders were retained, and relationship indicators were created to support analysis-specific filtering. Orders will only be excluded when the required related record is essential to the metric being calculated.

***Validation:*** The cleaned orders table still contains all 99,441 original orders, including 775 without items, one without payment, and 768 without reviews.

## 13. Final Cleaning Validation

Before exporting the processed datasets, row counts and critical cleaning rules are validated against the original raw data.

In [18]:
row_validation_records = []

for dataset_name in cleaned_datasets:
    raw_rows = len(raw_datasets[dataset_name])
    cleaned_rows = len(cleaned_datasets[dataset_name])

    row_validation_records.append({
        "dataset": dataset_name,
        "raw_rows": raw_rows,
        "cleaned_rows": cleaned_rows,
        "row_difference": cleaned_rows - raw_rows
    })

row_validation_summary = pd.DataFrame(
    row_validation_records
)

row_validation_summary.style.format({
    "raw_rows": "{:,.0f}",
    "cleaned_rows": "{:,.0f}",
    "row_difference": "{:+,.0f}"
})

,dataset,raw_rows,cleaned_rows,row_difference
0,customers,"99,441","99,441",+0
1,geolocation,"1,000,163","738,332","-261,831"
2,order_items,"112,650","112,650",+0
3,payments,"103,886","103,886",+0
4,reviews,"99,224","99,224",+0
5,orders,"99,441","99,441",+0
6,products,"32,951","32,951",+0
7,sellers,"3,095","3,095",+0
8,category_translation,71,74,+3


***Observation:*** All transactional and business-entity datasets retained their original row counts. The only reductions occurred in geolocation, where 261,831 exact duplicates were removed, while the category translation table gained three documented reference records.

***Validation:*** The row-count differences match the intended cleaning actions. No customer, order, item, payment, review, product, or seller records were deleted.

In [20]:
validation_checks = pd.DataFrame([
    {
        "validation_check": "Exact geolocation duplicates remaining",
        "result": cleaned_datasets["geolocation"].duplicated().sum(),
        "expected": 0
    },
    {
        "validation_check": "Untranslated product categories remaining",
        "result": len(
            set(
                cleaned_datasets["products"][
                    "product_category_name"
                ]
            )
            - set(
                cleaned_datasets["category_translation"][
                    "product_category_name"
                ]
            )
        ),
        "expected": 0
    },
    {
        "validation_check": "Invalid customer ZIP-code lengths",
        "result": (
            cleaned_datasets["customers"][
                "customer_zip_code_prefix"
            ].str.len() != 5
        ).sum(),
        "expected": 0
    },
    {
        "validation_check": "Invalid seller ZIP-code lengths",
        "result": (
            cleaned_datasets["sellers"][
                "seller_zip_code_prefix"
            ].str.len() != 5
        ).sum(),
        "expected": 0
    },
    {
        "validation_check": "Invalid geolocation ZIP-code lengths",
        "result": (
            cleaned_datasets["geolocation"][
                "geolocation_zip_code_prefix"
            ].str.len() != 5
        ).sum(),
        "expected": 0
    },
    {
        "validation_check": "Zero payment installments remaining",
        "result": (
            cleaned_datasets["payments"][
                "payment_installments"
            ] == 0
        ).sum(),
        "expected": 0
    },
    {
        "validation_check": "not_defined payment types remaining",
        "result": (
            cleaned_datasets["payments"][
                "payment_type"
            ] == "not_defined"
        ).sum(),
        "expected": 0
    },
    {
        "validation_check": "Non-positive product weights remaining",
        "result": (
            cleaned_datasets["products"][
                "product_weight_g"
            ].dropna() <= 0
        ).sum(),
        "expected": 0
    }
])

validation_checks["status"] = np.where(
    validation_checks["result"]
    == validation_checks["expected"],
    "Passed",
    "Failed"
)

validation_checks

,validation_check,result,expected,status
0,Exact geolocation duplicates remaining,0,0,Passed
1,Untranslated product categories remaining,0,0,Passed
2,Invalid customer ZIP-code lengths,0,0,Passed
3,Invalid seller ZIP-code lengths,0,0,Passed
4,Invalid geolocation ZIP-code lengths,0,0,Passed
5,Zero payment installments remaining,0,0,Passed
6,not_defined payment types remaining,0,0,Passed
7,Non-positive product weights remaining,0,0,Passed


## 14. Export Processed Data

The validated working copies are exported to `data/processed/`. The original files in `data/raw/` remain unchanged.

In [21]:
processed_filenames = {
    "customers": "customers_clean.csv",
    "geolocation": "geolocation_clean.csv",
    "order_items": "order_items_clean.csv",
    "payments": "payments_clean.csv",
    "reviews": "reviews_clean.csv",
    "orders": "orders_clean.csv",
    "products": "products_clean.csv",
    "sellers": "sellers_clean.csv",
    "category_translation": "category_translation_clean.csv"
}

export_records = []

for dataset_name, output_filename in processed_filenames.items():
    output_path = PROCESSED_DATA_DIR / output_filename

    cleaned_datasets[dataset_name].to_csv(
        output_path,
        index=False,
        date_format="%Y-%m-%d %H:%M:%S"
    )

    export_records.append({
        "dataset": dataset_name,
        "output_file": output_filename,
        "rows_exported": len(cleaned_datasets[dataset_name]),
        "file_created": output_path.exists()
    })

cleaning_log_df = pd.DataFrame(cleaning_log)

cleaning_log_path = (
    PROCESSED_DATA_DIR / "cleaning_log.csv"
)

cleaning_log_df.to_csv(
    cleaning_log_path,
    index=False
)

export_summary = pd.DataFrame(export_records)

export_summary

,dataset,output_file,rows_exported,file_created
0,customers,customers_clean.csv,99441,True
1,geolocation,geolocation_clean.csv,738332,True
2,order_items,order_items_clean.csv,112650,True
3,payments,payments_clean.csv,103886,True
4,reviews,reviews_clean.csv,99224,True
5,orders,orders_clean.csv,99441,True
6,products,products_clean.csv,32951,True
7,sellers,sellers_clean.csv,3095,True
8,category_translation,category_translation_clean.csv,74,True


In [22]:
print("Cleaned datasets exported:", len(export_summary))
print("All files created:", export_summary["file_created"].all())
print("Cleaning actions documented:", len(cleaning_log_df))
print("Cleaning log created:", cleaning_log_path.exists())

Cleaned datasets exported: 9
All files created: True
Cleaning actions documented: 26
Cleaning log created: True


## Conclusion

***Overall result:*** Nine cleaned and validated datasets were produced without modifying the original raw files. All transactional and entity records were preserved, except for exact duplicate geolocation rows, and three documented reference categories were added.

***Key transformations:*** Timestamp columns were converted to datetime, missing review text was documented with indicators, product categories and translations were completed, invalid product weights were converted to null, payment anomalies were flagged, ZIP-code prefixes were standardised, and timestamp and relationship-quality indicators were created.

***Validation:*** All final cleaning rules passed. The processed datasets and cleaning log were exported successfully to `data/processed/`.

***Next step:*** The cleaned relational tables will be integrated into analysis-ready order-, customer-, and seller-level datasets in `03_data_integration.ipynb`.